In [119]:
import os

print("当前工作目录:", os.getcwd())
print("当前文件所在目录:", os.path.dirname(os.path.abspath('__file__')))

# 列出上级目录的内容
parent_dir = os.path.dirname(os.getcwd())
print("上级目录:", parent_dir)
print("上级目录内容:", os.listdir(parent_dir))

当前工作目录: D:\programming\python_script\socio-physical system for shelter
当前文件所在目录: D:\programming\python_script\socio-physical system for shelter
上级目录: D:\programming\python_script
上级目录内容: ['ABM_with_GPU', 'AgentSociety', 'ancient_map_flood', 'C-LEARNING_VISULIZATION_GRAPH', 'FLAMEGPU2', 'FLAMEGPU2-tutorial-python', 'FLAMEGPU2_python_sugarscape_tutorial', 'geoai_related', 'LITTLE', 'network_select', 'PandemicLLM', 'scrapy', 'SHELTER_FLAMEGPU2', 'shelter_gravity_model', 'sleep_health_lifestyle', 'socio-physical system for shelter', 'test.ipynb', 'test.py', 'testforgit', 'transfer']


In [120]:
import sys
import os

# 切换到你的项目根目录
project_root = r"D:\programming\python_script\socio-physical system for shelter"
os.chdir(project_root)

# 添加 data/output 目录到 Python 路径
sys.path.append('data/output')


## define model

In [121]:
#这个test.py是用来测试pyflame的可视性的

from pyflamegpu import *
import pyflamegpu.codegen
import sys

# Define some useful constants



# Define the FLAME GPU model: 这个可以在后续的可视化窗口改名字
model = pyflamegpu.ModelDescription("First test using default visualization")


## messages setting

In [122]:

# Define a message of type MessageSpatial2D named location
# MessageSpatial2D: Each agent outputs a message at a specific location in 2D space
# agents only read messages located close to a particular search origin（搜素的中心点）.
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list

message.setMin(0, 0,0)
message.setMax(500, 500,100)
message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")



In [123]:
stairwell_message = model.newMessageSpatial3D("stairwell_location")
# Configure the message list

stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500,100)
stairwell_message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
stairwell_message.newVariableID("id")
stairwell_message.newVariableInt("building_id")

## agent_variables_definition

In [124]:

# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)
student_agent.newVariableInt("target_stairwell_id", -1)
student_agent.newVariableFloat("target_stairwell_x")
student_agent.newVariableFloat("target_stairwell_y")
student_agent.newVariableInt("evacuate_status", 1)
student_agent.newVariableInt("zigzag_dir", 1)


stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableInt("building_id")
stairwell_agent.newVariableFloat("z")



## environment setting

In [125]:

# Define environment properties
env = model.Environment()
env.newPropertyUInt("AGENT_COUNT", 10000)
env.newPropertyFloat("ENV_WIDTH", 500)
env.newPropertyFloat("repulse", 0.05)


## agent function

In [126]:
@pyflamegpu.agent_function
def stairwell_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("building_id", pyflamegpu.getVariableInt("building_id"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE

In [127]:
@pyflamegpu.agent_function
def set_target_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    # Get this agent's x, y, z variables
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    target_stairwell_id = pyflamegpu.getVariableInt("target_stairwell_id")

    min_dist = 100000
    if target_stairwell_id == -1:
        for message in message_in(x,y,z):
            # Process the message's variables e.g.
            if message.getVariableInt("building_id") == pyflamegpu.getVariableInt("building_id"):
                #找到距离最近的楼梯
                # 找到距离最近的楼梯

                stairwell_x = message.getVariableFloat("x")
                stairwell_y = message.getVariableFloat("y")

                # 计算欧氏距离
                dx = stairwell_x - x
                dy = stairwell_y - y

                dist = math.sqrtf(dx*dx + dy*dy)
                if dist < min_dist:
                    min_dist = dist
                    nearest_stairwell_id = message.getVariableInt("id")
                    # 记录最近楼梯的坐标
                    pyflamegpu.setVariableInt("target_stairwell_id", nearest_stairwell_id)
                    pyflamegpu.setVariableFloat("target_stairwell_x", stairwell_x)
                    pyflamegpu.setVariableFloat("target_stairwell_y", stairwell_y)

            #设置目标楼梯
    return pyflamegpu.ALIVE

In [128]:
import math

My tips:
https://github.com/FLAMEGPU/FLAMEGPU2/discussions/1307

In [129]:
@pyflamegpu.agent_function
def move_to_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    target_stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    target_stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    
    delta_x = target_stairwell_x - x
    delta_y = target_stairwell_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 1.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + step_x
        next_y = y + step_y
    else:
        next_x =  target_stairwell_x
        next_y =  target_stairwell_y
        pyflamegpu.setVariableInt("evacuate_status", 2)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    
    message_out.setLocation(next_x, next_y, pyflamegpu.getVariableFloat("z"))


    return pyflamegpu.ALIVE

In [130]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_1() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 1

In [131]:

@pyflamegpu.agent_function
def output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE


In [132]:
# z字形下楼的算法
# 假设每层楼高3，行人和stairwell的x、y初始一致
# 通过y,z索引，z字形移动，每次移动一小步，遇到转折点y反向

@pyflamegpu.agent_function
def down_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    """
    行人在stairwell点z字形下降，每层楼高3
    通过y,z索引，z字形移动
    """
    # 获取当前位置
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    # 获取目标stairwell的x,y
    stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    # 获取当前z字形方向（1为y正向，-1为y负向）
    if not pyflamegpu.getVariableInt("zigzag_dir"):
        pyflamegpu.setVariableInt("zigzag_dir", 1)

    dire = pyflamegpu.getVariableInt("zigzag_dir")
    # 每步y方向移动距离
    step_y = 0.5
    # 每步z方向下降距离
    step_z = 0.2
    # z字形的y范围（比如以stairwell_y为中心，上下各1.5）
    y_min = stairwell_y - 1.5
    y_max = stairwell_y + 1.5
    # 计算下一步y
    next_y = y + dire * step_y
    # 判断是否到达边界，若到达则反向
    if next_y > y_max:
        next_y = y_max
        dire = -1
    elif next_y < y_min:
        next_y = y_min
        dire = 1
    # 计算下一步z
    next_z = z - step_z
    # 判断是否到达下一层（z是否小于目标z）
    # 假设目标z为0
    if next_z < 0:
        next_z = 0
        pyflamegpu.setVariableInt("evacuate_status", 3)
    # 更新变量
    pyflamegpu.setVariableFloat("y", next_y)
    pyflamegpu.setVariableFloat("z", next_z)
    pyflamegpu.setVariableInt("zigzag_dir", dire)
    # x保持不变
    pyflamegpu.setVariableFloat("x", stairwell_x)
    # 输出当前位置
    message_out.setLocation(stairwell_x, next_y, next_z)
    return pyflamegpu.ALIVE

    

In [133]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_2() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 2

## function write in

In [134]:

# translate the agent functions from Python to C++
output_func_translated = pyflamegpu.codegen.translate(output_message)
stairwell_output_func_translated = pyflamegpu.codegen.translate(stairwell_output_message)
set_target_stairwell_func_translated = pyflamegpu.codegen.translate(set_target_stairwell)
move_to_stairwell_func_translated = pyflamegpu.codegen.translate(move_to_stairwell)
down_stairwell_func_translated = pyflamegpu.codegen.translate(down_stairwell)

#conditional function translate
eva_state_is_1_func_translated = pyflamegpu.codegen.translate(eva_state_is_1)
eva_state_is_2_func_translated = pyflamegpu.codegen.translate(eva_state_is_2)


# Setup the two agent functions
out_fn = student_agent.newRTCFunction("output_message", output_func_translated)
out_fn.setMessageOutput("location")

stairwell_output_fn = stairwell_agent.newRTCFunction("stairwell_output_message", stairwell_output_func_translated)
stairwell_output_fn.setMessageOutput("stairwell_location")

set_target_stairwell_fn = student_agent.newRTCFunction("set_target_stairwell", set_target_stairwell_func_translated)
set_target_stairwell_fn.setMessageInput("stairwell_location")

move_to_stairwell_fn = student_agent.newRTCFunction("move_to_stairwell", move_to_stairwell_func_translated)
move_to_stairwell_fn.setMessageOutput("location")
move_to_stairwell_fn.setRTCFunctionCondition(eva_state_is_1_func_translated)

down_stairwell_fn = student_agent.newRTCFunction("down_stairwell", down_stairwell_func_translated)
down_stairwell_fn.setMessageOutput("location")
down_stairwell_fn.setRTCFunctionCondition(eva_state_is_2_func_translated)

# Message input depends on output
out_fn.dependsOn(stairwell_output_fn)
set_target_stairwell_fn.dependsOn(stairwell_output_fn)
move_to_stairwell_fn.dependsOn(set_target_stairwell_fn)
move_to_stairwell_fn.dependsOn(out_fn)
down_stairwell_fn.dependsOn(move_to_stairwell_fn)


# 添加学生代理类型
# 基于data/output/flamegpu_init_code.py的学生代理初始化


# Dependency specification
# Output is the root of our graph
model.addExecutionRoot(stairwell_output_fn) 
model.generateLayers()



## simulation creation

In [135]:

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)



## initialization

In [136]:


from flamegpu_init_code import initialize_student_agent_population

# 初始化学生代理种群
initialize_student_agent_population(model, cuda_model)

# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1) 
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanFloat("drift")
step_log_cfg.agent("student_agent").logMeanFloat("x")
step_log_cfg.agent("student_agent").logMeanFloat("y")


cuda_model.initialise(sys.argv)


# Attach the logging config
cuda_model.setStepLog(step_log_cfg)


初始化 6120 个学生代理个体


学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成


## visualization

In [137]:

WIDTH=500

# Only run this block if pyflamegpu was built with visualisation support
if pyflamegpu.VISUALISATION:
    # Create visualisation
    m_vis = cuda_model.getVisualisation()
    # Set the initial camera location and speed
    INIT_CAM = WIDTH / 2
    m_vis.setInitialCameraTarget(270, 205, 0)
    m_vis.setInitialCameraLocation(240, 100, 100)
    m_vis.setCameraSpeed(0.01)
    m_vis.setSimulationSpeed(10)
    # Add "point" agents to the visualisation

    
    # Add "student_agent" agents to the visualisation
    student_agt = m_vis.addAgent("student_agent")
    student_agt.setModel(pyflamegpu.ICOSPHERE);
    student_agt.setModelScale(1/1.0);
    # Mark the environment bounds.

    stairwell_agt = m_vis.addAgent("stairwell_agent")
    stairwell_agt.setModel(pyflamegpu.ICOSPHERE);
    stairwell_agt.setModelScale(1/0.5);
    stairwell_agt.setColor(pyflamegpu.RED);
    
    pen = m_vis.newPolylineSketch(1, 1, 1, 0.2)
    pen.addVertex(275, 637, 0) # 起始点
    pen.addVertex(69, 510, 0)
    pen.addVertex(0, 301, 0)
    pen.addVertex(1, 167, 0)
    pen.addVertex(29, 142, 0)
    pen.addVertex(57, 98, 0)
    pen.addVertex(118, 67, 0)
    pen.addVertex(109, 24, 0)
    pen.addVertex(287, 0, 0)
    pen.addVertex(286, 45, 0)
    pen.addVertex(405, 154, 0)
    pen.addVertex(435, 131, 0)
    pen.addVertex(436, 72, 0)
    pen.addVertex(467, 41, 0)
    pen.addVertex(501, 35, 0)
    pen.addVertex(543, 47, 0)
    pen.addVertex(275, 637, 0) # 闭合点 
    # Open the visualiser window 
    m_vis.activate()

# Run the simulation
for i in range(200):
    cuda_model.step()



if pyflamegpu.VISUALISATION:
    # Keep the visualisation window active after the simulation has completed
    m_vis.join()

## data collection

In [138]:
import numpy as np

out_pop = pyflamegpu.AgentVector(model.Agent("student_agent"))
cuda_model.getPopulationData(out_pop)

# 创建结构化数组
dtype = [('target_stairwell_id', 'i4'),('building_id', 'i4'),('x','f4'),('y','f4'),('z', 'f4')]
agent_array = np.array(
    [(agent.getVariableInt("target_stairwell_id"), agent.getVariableInt("building_id"), agent.getVariableFloat("x"),agent.getVariableFloat("y"),agent.getVariableFloat("z")) 
     for agent in out_pop],
    dtype=dtype
)

agent_array

array([(16,  5, 304.47333, 164.60149, 0.        ),
       (16,  5, 304.47333, 164.60149, 0.        ),
       (16,  5, 304.47333, 164.60149, 0.        ), ...,
       (24, 10, 205.39262, 471.82306, 3.3999243 ),
       ( 1, 10, 171.36388, 449.6318 , 3.3999243 ),
       ( 1, 10, 171.36388, 449.6318 , 0.39993525)],
      dtype=[('target_stairwell_id', '<i4'), ('building_id', '<i4'), ('x', '<f4'), ('y', '<f4'), ('z', '<f4')])

In [139]:
out_pop_stairwell = pyflamegpu.AgentVector(model.Agent("stairwell_agent"))
cuda_model.getPopulationData(out_pop_stairwell)

# 创建结构化数组
dtype1 = [('building_id', 'i4'),('x','f4'),('y','f4'),('z', 'f4'),('stairwell_id', 'i4')]
agent_array1 = np.array(
    [(stairwell_agent.getVariableInt("building_id"), stairwell_agent.getVariableFloat("x"),stairwell_agent.getVariableFloat("y"),stairwell_agent.getVariableFloat("z"),stairwell_agent.getVariableInt("stairwell_id")) 
     for stairwell_agent in out_pop_stairwell],
    dtype=dtype1
) 

agent_array1

array([(10, 171.36388, 451.1318  , 0.,  0),
       (11, 187.40019, 422.72263 , 0.,  1),
       (11, 185.35193, 403.4037  , 0.,  2),
       (12, 292.73672, 513.2608  , 0.,  3),
       (13, 275.96817, 489.26135 , 0.,  4),
       (13, 313.38123, 451.64853 , 0.,  5),
       ( 1, 338.08554, 305.69458 , 0.,  6),
       ( 2, 353.96848, 291.7906  , 0.,  7),
       ( 2, 401.3628 , 299.5717  , 0.,  8),
       ( 1, 345.0179 , 341.7112  , 0.,  9),
       ( 2, 350.91205, 258.75668 , 0., 10),
       ( 2, 392.2505 , 255.45209 , 0., 11),
       ( 0, 267.1187 , 263.7444  , 0., 12),
       ( 4, 298.45358, 252.1687  , 0., 13),
       ( 3, 303.60367, 211.91917 , 0., 14),
       ( 5, 304.47333, 164.10149 , 0., 15),
       ( 5, 354.6269 , 156.37993 , 0., 16),
       ( 5, 323.09534, 108.672104, 0., 17),
       ( 6, 428.95746, 179.99527 , 0., 18),
       ( 8, 446.40863,  86.777985, 0., 19),
       ( 9, 510.3985 ,  84.81714 , 0., 20),
       ( 9, 498.04996,  60.32895 , 0., 21),
       ( 6, 464.56763, 181.44775